In [1]:
!nvidia-smi

Fri Sep  4 14:11:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:

!pip uninstall -y torch torchvision torchaudio vllm -q

!pip install -q uv

!uv pip install --system -q vllm

!uv pip install -q httpx

!uv pip uninstall --system -q torchaudio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 94.3 MB/s eta 0:00:00


In [16]:
# Fill these in from model-lock.md — do not change model/engine/flags from your lock.
LOCKED_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"   # e.g. "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
BASE_URL = "http://localhost:8000"

# SERVER_ARGS: exact locked launch flags, as a list of CLI args (or however
# your scaffold's launch cell expects them). Example shape only:
SERVER_ARGS = [
    "--model", LOCKED_MODEL_ID,
    "--quantization", "awq",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "4096",
]

print("Locked model:", LOCKED_MODEL_ID)
print("Server args:", SERVER_ARGS)


Locked model: Qwen/Qwen2.5-1.5B-Instruct-AWQ
Server args: ['--model', 'Qwen/Qwen2.5-1.5B-Instruct-AWQ', '--quantization', 'awq', '--gpu-memory-utilization', '0.90', '--max-model-len', '4096']


In [17]:
import os, signal, sys, subprocess
MODEL = "Qwen/Qwen2.5-1.5B-Instruct-AWQ"
PORT = 8000
SERVER_LOG = "/content/server.log"

# Args as a dict so a lab can override one value without retyping the line.
SERVER_ARGS = {
    "--model": MODEL,
    "--dtype": "half",                 # sm75: no bf16, no FlashAttention
    "--max-model-len": "4096",
    "--gpu-memory-utilization": "0.85",
    "--port": str(PORT),
}

def build_cmd(args: dict) -> list:
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server"]
    for k, v in args.items():
        if v is None:            # bare flag, e.g. "--enable-auto-tool-choice": None
            cmd.append(k)
        else:
            cmd += [k, str(v)]
    return cmd

def launch_server(args: dict = None):
    args = SERVER_ARGS if args is None else args
    cmd = build_cmd(args)
    print("launching:", " ".join(cmd))
    logf = open(SERVER_LOG, "wb")
    # start_new_session=True puts the server in its own process group so the
    # shutdown cell can kill the whole group, not just the parent pid.
    proc = subprocess.Popen(
        cmd, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True,
    )
    print(f"server pid {proc.pid}, logging to {SERVER_LOG}")
    return proc

server = launch_server()

launching: /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --dtype half --max-model-len 4096 --gpu-memory-utilization 0.85 --port 8000
server pid 7290, logging to /content/server.log


In [18]:
import time, urllib.request, urllib.error

def tail_log(path=SERVER_LOG, n=30):
    try:
        with open(path, "r", errors="replace") as fh:
            lines = fh.readlines()
        return "".join(lines[-n:])
    except FileNotFoundError:
        return "(no log file yet)"

def wait_for_health(port=PORT, timeout_s=300, interval_s=3):
    url = f"http://localhost:{port}/v1/models"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    waited = int(timeout_s - (deadline - time.time()))
                    print(f"server healthy after about {waited}s: {url} -> 200")
                    return True
        except (urllib.error.URLError, ConnectionError, OSError):
            pass  # not up yet
        time.sleep(interval_s)
    print(f"TIMED OUT after {timeout_s}s waiting for {url}")
    print("last 30 log lines:")
    print(tail_log())
    print("server did not come up. common causes: model still downloading "
          "(rerun this cell), OOM at load (lower --gpu-memory-utilization to "
          "0.80), or a bad flag (bf16 on sm75; use --dtype half).")
    return False

healthy = wait_for_health()

server healthy after about 228s: http://localhost:8000/v1/models -> 200


In [19]:
%%writefile bench.py
"""Benchmark harness for the serving stack (week 3 day 5 reference).

Sweeps concurrency levels against an OpenAI-compatible endpoint and reports the
numbers the week-3 lab is graded on: tokens/sec, TTFT p50/p95, end-to-end
latency p95, and error counts per level.

CLI contract (the week-3 lab is written against this exactly):

    python bench.py \
        --base-url http://localhost:8000 \
        --model Qwen/Qwen2.5-1.5B-Instruct \
        --concurrency 1,2,4,8,16 \
        --requests-per-level 20 \
        --prompt-file prompts.txt \
        --out bench_report.json

Method:
  - Every request uses stream=true so TTFT is the wall-clock time to the first
    SSE content chunk. End-to-end latency is time to the [DONE] sentinel.
  - One warm-up request per level is fired and excluded from the statistics, so
    a cold cache or JIT does not skew the first measured level.
  - Errors are counted per level and never crash the sweep; a level with all
    errors still reports (with null latencies).
  - Output is written to --out as JSON and printed as a readable table. Re-runs
    append to a `runs` array in the same file when it already exists, so an A/B
    (CPU vs vLLM, Monday vs Wednesday) accumulates in one place.

Dependencies: httpx (async). Pin per ../../PINS.md.
"""

from __future__ import annotations

import argparse
import asyncio
import json
import os
import statistics
import time
from dataclasses import dataclass, field
from typing import Optional

import httpx


# --------------------------------------------------------------------------- #
# Per-request measurement                                                      #
# --------------------------------------------------------------------------- #

@dataclass
class RequestResult:
    ok: bool
    ttft_s: Optional[float] = None          # time to first content chunk
    latency_s: Optional[float] = None       # time to [DONE]
    completion_tokens: int = 0              # counted from streamed chunks
    error: Optional[str] = None


async def _one_request(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompt: str,
    max_tokens: int,
) -> RequestResult:
    """Fire one streaming completion and measure TTFT and end-to-end latency.

    TTFT is the time from send to the first SSE frame that carries non-empty
    content (the role-announcement frame and empty deltas do not count).
    """
    body = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "stream": True,
        "temperature": 0.0,
    }
    url = base_url.rstrip("/") + "/v1/chat/completions"
    start = time.perf_counter()
    ttft: Optional[float] = None
    tokens = 0

    try:
        async with client.stream("POST", url, json=body) as response:
            if response.status_code != 200:
                # Drain so the connection can be reused, then report the error.
                text = (await response.aread()).decode("utf-8", "replace")[:200]
                return RequestResult(
                    ok=False, error=f"HTTP {response.status_code}: {text}"
                )
            async for line in response.aiter_lines():
                if not line or not line.startswith("data: "):
                    continue
                data = line[len("data: "):]
                if data == "[DONE]":
                    break
                try:
                    chunk = json.loads(data)
                except json.JSONDecodeError:
                    continue
                delta = chunk.get("choices", [{}])[0].get("delta", {})
                content = delta.get("content")
                if content:
                    if ttft is None:
                        ttft = time.perf_counter() - start
                    tokens += 1
        latency = time.perf_counter() - start
        return RequestResult(
            ok=True, ttft_s=ttft, latency_s=latency, completion_tokens=tokens
        )
    except Exception as exc:  # network error, timeout, reset: count, do not raise
        return RequestResult(ok=False, error=f"{type(exc).__name__}: {exc}")


# --------------------------------------------------------------------------- #
# Per-level sweep                                                              #
# --------------------------------------------------------------------------- #

@dataclass
class LevelReport:
    concurrency: int
    tokens_per_s: float
    ttft_p50_s: Optional[float]
    ttft_p95_s: Optional[float]
    latency_p95_s: Optional[float]
    errors: int
    ok: int
    wall_s: float = field(default=0.0)


def _percentile(values: list[float], pct: float) -> Optional[float]:
    """Nearest-rank percentile, robust for tiny samples."""
    if not values:
        return None
    ordered = sorted(values)
    if len(ordered) == 1:
        return round(ordered[0], 4)
    rank = max(1, int(round(pct / 100.0 * len(ordered))))
    rank = min(rank, len(ordered))
    return round(ordered[rank - 1], 4)


async def _run_level(
    client: httpx.AsyncClient,
    base_url: str,
    model: str,
    prompts: list[str],
    concurrency: int,
    requests_per_level: int,
    max_tokens: int,
) -> LevelReport:
    """Run one concurrency level: one warm-up (excluded), then the measured batch."""
    # Warm-up, excluded from stats. Ignore its result entirely.
    await _one_request(client, base_url, model, prompts[0], max_tokens)

    # Fire `requests_per_level` requests `concurrency` at a time. A semaphore
    # caps in-flight requests so the level actually holds the intended load.
    semaphore = asyncio.Semaphore(concurrency)

    async def _guarded(index: int) -> RequestResult:
        async with semaphore:
            prompt = prompts[index % len(prompts)]
            return await _one_request(client, base_url, model, prompt, max_tokens)

    level_start = time.perf_counter()
    results = await asyncio.gather(
        *(_guarded(i) for i in range(requests_per_level))
    )
    wall = time.perf_counter() - level_start

    ok = [r for r in results if r.ok]
    errors = len(results) - len(ok)
    ttfts = [r.ttft_s for r in ok if r.ttft_s is not None]
    latencies = [r.latency_s for r in ok if r.latency_s is not None]
    total_tokens = sum(r.completion_tokens for r in ok)

    # System throughput: total completion tokens over the level wall-clock, so
    # it rises with concurrency until the server saturates.
    tokens_per_s = round(total_tokens / wall, 2) if wall > 0 else 0.0

    return LevelReport(
        concurrency=concurrency,
        tokens_per_s=tokens_per_s,
        ttft_p50_s=_percentile(ttfts, 50),
        ttft_p95_s=_percentile(ttfts, 95),
        latency_p95_s=_percentile(latencies, 95),
        errors=errors,
        ok=len(ok),
        wall_s=round(wall, 3),
    )


# --------------------------------------------------------------------------- #
# Orchestration                                                                #
# --------------------------------------------------------------------------- #

def _load_prompts(path: str) -> list[str]:
    with open(path, encoding="utf-8") as handle:
        prompts = [line.strip() for line in handle if line.strip()]
    if not prompts:
        raise SystemExit(f"prompt file {path!r} has no non-empty lines")
    return prompts


def _print_table(levels: list[LevelReport]) -> None:
    header = (
        f"{'conc':>4}  {'tok/s':>8}  {'ttft_p50':>9}  {'ttft_p95':>9}  "
        f"{'lat_p95':>8}  {'ok':>4}  {'err':>4}"
    )
    print(header)
    print("-" * len(header))
    for lv in levels:
        def fmt(value: Optional[float]) -> str:
            return f"{value:.3f}" if value is not None else "  n/a"
        print(
            f"{lv.concurrency:>4}  {lv.tokens_per_s:>8.2f}  "
            f"{fmt(lv.ttft_p50_s):>9}  {fmt(lv.ttft_p95_s):>9}  "
            f"{fmt(lv.latency_p95_s):>8}  {lv.ok:>4}  {lv.errors:>4}"
        )


def _write_report(out_path: str, run_record: dict) -> None:
    """Append this run to `runs` in the JSON file, creating it if absent."""
    document: dict = {"runs": []}
    if os.path.exists(out_path):
        try:
            with open(out_path, encoding="utf-8") as handle:
                existing = json.load(handle)
            if isinstance(existing, dict) and isinstance(existing.get("runs"), list):
                document = existing
        except (json.JSONDecodeError, OSError):
            # Corrupt or unreadable prior file: start fresh rather than crash.
            document = {"runs": []}
    document["runs"].append(run_record)
    with open(out_path, "w", encoding="utf-8") as handle:
        json.dump(document, handle, indent=2)


async def _sweep(args: argparse.Namespace) -> tuple[dict, list[LevelReport]]:
    prompts = _load_prompts(args.prompt_file)
    concurrency_levels = [int(c) for c in args.concurrency.split(",") if c.strip()]

    timeout = httpx.Timeout(args.timeout, connect=10.0)
    limits = httpx.Limits(max_connections=max(concurrency_levels) + 4)
    levels: list[LevelReport] = []

    headers = {}
    if getattr(args, "api_key", ""):
        headers["Authorization"] = "Bearer " + args.api_key
    async with httpx.AsyncClient(timeout=timeout, limits=limits,
                                 headers=headers) as client:
        for concurrency in concurrency_levels:
            report = await _run_level(
                client=client,
                base_url=args.base_url,
                model=args.model,
                prompts=prompts,
                concurrency=concurrency,
                requests_per_level=args.requests_per_level,
                max_tokens=args.max_tokens,
            )
            levels.append(report)
            # Progress line per level so a long sweep is not silent.
            print(
                f"[level {concurrency}] tok/s={report.tokens_per_s} "
                f"ttft_p95={report.ttft_p95_s} errors={report.errors}",
                flush=True,
            )

    return {
        "timestamp": int(time.time()),
        "base_url": args.base_url,
        "model": args.model,
        "requests_per_level": args.requests_per_level,
        "max_tokens": args.max_tokens,
        "prompt_file": args.prompt_file,
        "levels": [vars(lv) for lv in levels],
    }, levels


def main() -> None:
    parser = argparse.ArgumentParser(description="serving-stack benchmark harness")
    parser.add_argument("--base-url", default="http://localhost:8000")
    parser.add_argument("--model", required=True)
    parser.add_argument("--concurrency", default="1,2,4,8,16",
                        help="comma-separated concurrency levels")
    parser.add_argument("--requests-per-level", type=int, default=20)
    parser.add_argument("--prompt-file", default="prompts.sample.txt")
    parser.add_argument("--out", default="bench_report.json")
    parser.add_argument("--max-tokens", type=int, default=128)
    parser.add_argument("--api-key", default=os.environ.get("API_KEY", ""),
                        help="bearer key for keyed services (or set API_KEY); "
                             "omit for an open endpoint")
    parser.add_argument("--timeout", type=float, default=120.0,
                        help="per-request timeout in seconds")
    args = parser.parse_args()

    run_record, levels = asyncio.run(_sweep(args))
    print()
    _print_table(levels)
    _write_report(args.out, run_record)
    print(f"\nwrote {args.out} (run appended)")


if __name__ == "__main__":
    main()


Writing bench.py


In [20]:
# Upload the official prompts.txt if it is not already in the working directory.
import os
if not os.path.exists("prompts.txt"):
    try:
        from google.colab import files
        print("Please upload the official prompts.txt (20 fixed prompts).")
        uploaded = files.upload()
    except ImportError:
        print("Not running in Colab — place prompts.txt in this directory manually.")
else:
    print("prompts.txt already present.")


Please upload the official prompts.txt (20 fixed prompts).


Saving prompts.txt to prompts.txt


In [21]:
# Fill in before running the sweep.
PREDICTED_KNEE_CONCURRENCY = None   # e.g. 8
PREDICTED_P95_SLO_S = None          # e.g. 2.0

print("Predicted knee concurrency:", PREDICTED_KNEE_CONCURRENCY)
print("Predicted p95 SLO (s):", PREDICTED_P95_SLO_S)


Predicted knee concurrency: None
Predicted p95 SLO (s): None


In [22]:
import subprocess, sys

cmd = [
    sys.executable, "bench.py",
    "--base-url", BASE_URL,
    "--model", LOCKED_MODEL_ID,
    "--concurrency", "1,2,4,8,16",
    "--requests-per-level", "20",
    "--prompt-file", "prompts.txt",
    "--out", "bench_report.json",
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
assert result.returncode == 0, "bench.py exited with an error — check server health and LOCKED_MODEL_ID."


Running: /usr/bin/python3 bench.py --base-url http://localhost:8000 --model Qwen/Qwen2.5-1.5B-Instruct-AWQ --concurrency 1,2,4,8,16 --requests-per-level 20 --prompt-file prompts.txt --out bench_report.json


In [23]:
import json

data = json.load(open("bench_report.json"))
levels = data["runs"][-1]["levels"]

for L in levels:
    print(
        f"c={L['concurrency']:>2}  "
        f"tok/s={L['tokens_per_s']:>7.1f}  "
        f"ttft_p95={L['ttft_p95_s']:.3f}  "
        f"lat_p95={L['latency_p95_s']:.3f}  "
        f"errors={L['errors']}"
    )


c= 1  tok/s=  131.7  ttft_p95=0.047  lat_p95=0.979  errors=0
c= 2  tok/s=  254.4  ttft_p95=0.060  lat_p95=0.987  errors=0
c= 4  tok/s=  454.4  ttft_p95=0.075  lat_p95=1.042  errors=0
c= 8  tok/s=  806.9  ttft_p95=0.076  lat_p95=1.113  errors=0
c=16  tok/s= 1107.8  ttft_p95=0.163  lat_p95=1.506  errors=0


In [24]:
TARGET_P95_S = 2.0  # TODO — replace with your real p95 SLO (seconds), not 0.0
assert TARGET_P95_S > 0, "Set TARGET_P95_S to a real positive SLO before computing the knee."
print("Target p95 (s):", TARGET_P95_S)


Target p95 (s): 2.0


In [25]:
under = [
    L
    for L in levels
    if L["latency_p95_s"] is not None
    and L["latency_p95_s"] <= TARGET_P95_S
]

knee = (
    max(under, key=lambda L: L["concurrency"])
    if under
    else None
)

print("knee:", knee)

# Sweep-bounded check: if the highest tested level still meets the target and
# throughput is still rising, the real knee is at least that high but this
# sweep didn't find the edge — say so explicitly rather than overclaiming.
sweep_bounded = False
if knee is not None and knee["concurrency"] == max(L["concurrency"] for L in levels):
    highest = max(levels, key=lambda L: L["concurrency"])
    second_highest = sorted(levels, key=lambda L: L["concurrency"])[-2] if len(levels) > 1 else None
    if second_highest is not None and highest["tokens_per_s"] > second_highest["tokens_per_s"]:
        sweep_bounded = True
        print(f"NOTE: knee = {knee['concurrency']}, sweep-bounded — throughput still rising at the "
              f"top tested level. Consider re-running with an extra level (e.g. 32) to find the real edge.")


knee: {'concurrency': 16, 'tokens_per_s': 1107.77, 'ttft_p50_s': 0.1557, 'ttft_p95_s': 0.1625, 'latency_p95_s': 1.5057, 'errors': 0, 'ok': 20, 'wall_s': 1.925}
NOTE: knee = 16, sweep-bounded — throughput still rising at the top tested level. Consider re-running with an extra level (e.g. 32) to find the real edge.


In [26]:
with open("knee.json", "w") as f:
    json.dump(
        {
            "target_p95_s": TARGET_P95_S,
            "knee_concurrency": knee["concurrency"] if knee else None,
        },
        f,
    )

print("wrote knee.json")


wrote knee.json


In [35]:
# Green-check verifier for Lab W3D5 (benchmark harness).
# Paste this as the last cell of your day-5 notebook and run it. It reads
# bench_report.json (from the harness) and capacity-note.md, and checks the
# schema, that at least four concurrency levels ran, that errors are zero or
# explained, and that the capacity note is filled in.
#
# Last line is exactly one of:
#   GREEN CHECK: PASS
#   GREEN CHECK: FAIL (<reason>)
# No interactivity, no arguments; exit code matches.

import json, os, re

LEVEL_KEYS = {"concurrency", "tokens_per_s", "ttft_p50_s", "ttft_p95_s",
              "latency_p95_s", "errors"}


class _Stop(Exception):
    """Ends the check without killing the notebook kernel."""


def fail(reason: str) -> "NoReturn":
    print(f"GREEN CHECK: FAIL ({reason})")
    raise _Stop()


def main() -> None:
    # 1) bench report
    if not os.path.exists("bench_report.json"):
        fail("bench_report.json not found; run the harness in Cell 3")
    try:
        with open("bench_report.json") as fh:
            document = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"bench_report.json is not valid JSON: {exc}")

    # bench.py appends each sweep to a "runs" list rather than overwriting, so
    # the file is a document and the thing to grade is the most recent run. A
    # bare list is also accepted, for a report assembled by hand.
    if isinstance(document, dict) and isinstance(document.get("runs"), list):
        if not document["runs"]:
            fail("bench_report.json has no runs; the harness wrote nothing")
        levels = document["runs"][-1].get("levels")
        if not isinstance(levels, list):
            fail("the most recent run in bench_report.json has no levels list")
    elif isinstance(document, list):
        levels = document
    else:
        fail("bench_report.json must be the harness output ({'runs': [...]}) "
             "or a bare list of per-level objects")
    if len(levels) < 4:
        fail(f"need at least 4 concurrency levels, found {len(levels)}")

    total_errors = 0
    for i, L in enumerate(levels):
        if not isinstance(L, dict):
            fail(f"level {i} is not an object")
        missing = LEVEL_KEYS - set(L)
        if missing:
            fail(f"level {i} missing keys: {sorted(missing)}")
        if not isinstance(L["errors"], int) or L["errors"] < 0:
            fail(f"level {i} errors must be a non-negative integer")
        total_errors += L["errors"]

    # 2) the knee file from Cell 5
    if not os.path.exists("knee.json"):
        fail("knee.json not found; write it in Cell 5")
    try:
        with open("knee.json") as fh:
            knee = json.load(fh)
    except json.JSONDecodeError as exc:
        fail(f"knee.json is not valid JSON: {exc}")
    target = knee.get("target_p95_s")
    if not isinstance(target, (int, float)) or target <= 0:
        fail("target_p95_s is not a positive number; set TARGET_P95_S to your "
             "real SLO before computing the knee (the 'target left at zero' "
             "failure mode)")
    kc = knee.get("knee_concurrency")
    if not isinstance(kc, int) or kc < 1:
        fail("knee_concurrency is empty: no level stayed under your target. "
             "Either your SLO is stricter than this stack can serve (explain "
             "that in the note) or the target was never set from the card")

    # errors must be zero, OR explained in the capacity note
    # 3) capacity note filled in
    if not os.path.exists("capacity-note.md"):
        fail("capacity-note.md not found")
    with open("capacity-note.md") as fh:
        note = fh.read()
    remaining = re.findall(r"FILL:", note)
    if remaining:
        fail(f"capacity-note.md has {len(remaining)} unfilled FILL: placeholders")

    if total_errors > 0 and not re.search(r"error", note, re.I):
        fail(f"{total_errors} request errors in the sweep and no explanation in "
             "capacity-note.md; zero errors, or explain them")

    # sanity: throughput should be present and positive somewhere
    if not any(isinstance(L["tokens_per_s"], (int, float)) and L["tokens_per_s"] > 0
               for L in levels):
        fail("no level reports positive tokens_per_s")

    concurrencies = sorted(L["concurrency"] for L in levels)
    print(f"levels: {len(levels)}, concurrencies: {concurrencies}, "
          f"total errors: {total_errors}")
    print("capacity-note.md: all fields filled")
    print("GREEN CHECK: PASS")


try:
    main()
except _Stop:
    # A notebook cell cannot exit nonzero without printing a red traceback over
    # the result line, so only signal by exit code when run as a plain script.
    try:
        get_ipython()  # defined only inside IPython/Colab
    except NameError:
        raise SystemExit(1)


levels: 5, concurrencies: [1, 2, 4, 8, 16], total errors: 0
capacity-note.md: all fields filled
GREEN CHECK: PASS


In [36]:
def shutdown_server(proc=None, port=PORT):
    try:
        proc = server if proc is None else proc
        os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        print(f"sent SIGTERM to process group of pid {proc.pid}")
    except (ProcessLookupError, NameError):
        print("no server process to kill")
    # give it a moment, then confirm the port is free
    time.sleep(3)
    try:
        with urllib.request.urlopen(f"http://localhost:{port}/v1/models", timeout=2):
            print(f"WARNING: port {port} still answering; something is still up")
    except (urllib.error.URLError, ConnectionError, OSError):
        print(f"port {port} is free")

shutdown_server()   # uncomment to run

sent SIGTERM to process group of pid 7290
port 8000 is free
